# 🧠 CivicEye — Export YOLO-World for On-Device AI (Google Colab)
# 
# Run each cell top-to-bottom with **Runtime → Run all** (or Shift+Enter per cell).
# This produces a single `.onnx` file you host on GitHub and drop into the app.
#
# **What it does:** loads YOLO-World (zero-shot detector), bakes your 11 civic
# labels into the weights, and exports a standard single-image ONNX that the
# CivicEye on-device engine already runs. No training required.

In [ ]:
# 1) Install Ultralytics (2 min)
!pip install -q ultralytics onnxruntime
import ultralytics
print("ultralytics", ultralytics.__version__)

In [ ]:
# 2) Load YOLO-World and set your 11 civic labels
from ultralytics import YOLOWorld

# Choose size:
#   yolov8n-worldv2.pt  → smallest/fastest  (~12 MB onnx fp32)
#   yolov8s-worldv2.pt  → best balance       (~23 MB onnx fp32)  <-- recommended
#   yolov8m-worldv2.pt  → most accurate      (~48 MB) — slower on phones
MODEL = "yolov8s-worldv2.pt"

model = YOLOWorld(MODEL)

# ⚠️ ORDER MATTERS — this exact order goes into VITE_ONDEVICE_YOLO_LABELS later.
CIVIC_LABELS = [
    "pothole",
    "broken-road",
    "garbage",
    "sidewalk",
    "manhole",
    "fallen-tree",
    "street-light",
    "water-leakage",
    "sewage",
    "illegal-dumping",
    "traffic-signal",
]
model.set_classes(CIVIC_LABELS)
print("classes frozen:", model.names)

In [ ]:
# 3) (Optional) Quick sanity test on a sample image
# Upload your own photo (left panel → 📁 Files → upload) or test on a URL.
import urllib.request
urllib.request.urlretrieve(
    "https://ultralytics.com/images/bus.jpg", "test.jpg"
)
results = model.predict("test.jpg", imgsz=640, conf=0.25)
results[0].show()
# It will print detections like "pothole 0.8x" — proves zero-shot detection works.

In [ ]:
# 4) Export to ONNX — fp32, NO simplify (critical fix)
# ⚠️ DO NOT use simplify=True for YOLO-World — onnxslim can break the
#    graph (you get "Graph output (output0) does not exist" in the app).
# The exported model is a STANDARD single-image YOLO: input [1,3,640,640],
# output [1, 15, 8400] (4 box values + 11 classes). Our app decodes this.
export_path = model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,        # fixed size → smaller + faster in the browser
    simplify=False,       # ← keep FALSE for YOLO-World
    opset=12,             # ← lower opset = safest for onnxruntime-web
)
print("ONNX saved at:", export_path)
print("Size MB:", round(__import__('os').path.getsize(export_path)/1e6, 1))

In [ ]:
# 5) (Optional) Shrink with fp16 — only if you know what you're doing
# fp16 halves the size BUT onnxruntime-web (WASM) can have trouble with
# fp16 weights in some browsers. If the app errors after switching to a
# fp16 file, go back to the fp32 one from cell 4.
# Ultralytics >= 8.4: quantize=16 for fp16.
try:
    export_path = model.export(
        format="onnx",
        imgsz=640,
        dynamic=False,
        simplify=False,
        opset=12,
        quantize=16,
    )
    print("fp16 ONNX saved at:", export_path)
    print("Size MB:", round(__import__('os').path.getsize(export_path)/1e6, 1))
except Exception as e:
    print("fp16 export failed — that's OK, keep the fp32 from cell 4:")
    print(" ", str(e)[:200])

In [ ]:
# 6) VERIFY the exported model (do this BEFORE hosting!)
# If this loads without error, your file is good. If it throws
# "Graph output (output0) does not exist" — the file is broken, re-export.
import onnxruntime as ort

try:
    sess = ort.InferenceSession(export_path, providers=["CPUExecutionProvider"])
    for inp in sess.get_inputs():
        print("INPUT :", inp.name, inp.shape, inp.type)
    for out in sess.get_outputs():
        print("OUTPUT:", out.name, out.shape, out.type)
    n_classes = 4 + 11  # your label count
    print("VERIFIED ✓ (expect output dims to include", n_classes, ")")
except Exception as e:
    print("❌ FILE IS BROKEN — re-export with simplify=False, opset=12")
    print("  ", str(e)[:200])

In [ ]:
# 7) Download your .onnx (two ways)
# A) Left sidebar → 📁 Files → find <name>.onnx → right-click → Download
#
# B) Or mount Google Drive and copy it there:
from google.colab import drive
drive.mount("/content/drive")
!cp {export_path} "/content/drive/MyDrive/"{export_path.split("/")[-1]}
print("Copied to Google Drive → MyDrive →", export_path.split("/")[-1])

# 8) Next: host it + wire it into the app
#
# 1) Upload the .onnx to a GitHub repo (e.g. Crepify/civiceye-models)
# 2) URL becomes:
#      https://cdn.jsdelivr.net/gh/Crepify/civiceye-models@main/<file>.onnx
# 3) In the app .env + Vercel:
#      VITE_ONDEVICE_YOLO_URL=<that URL>
#      VITE_ONDEVICE_YOLO_LABELS=pothole,broken-road,garbage,sidewalk,manhole,fallen-tree,street-light,water-leakage,sewage,illegal-dumping,traffic-signal
#      VITE_ONDEVICE_YOLO_SIZE=640
# 4) Rebuild → test a pothole photo → expect the 🖥️ on-device badge.
#
# More details: YOLO_WORLD_ONDEVICE.md in the project.